In [ ]:
# Setup: imports + helper de requisição
import requests
import pandas as pd

# get_json valida que a resposta é realmente JSON — evita o JSONDecodeError
# críptico quando uma API devolve HTML/vazio (como a do INMET às vezes faz).
def get_json(url, timeout=60):
    r = requests.get(url, timeout=timeout, headers={"Accept": "application/json"})
    r.raise_for_status()
    ct = r.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        raise ValueError(
            f"Resposta não-JSON de {url}\n"
            f"  status={r.status_code} content-type={ct}\n"
            f"  primeiros 300 chars: {r.text[:300]!r}"
        )
    return r.json()


In [ ]:
# 2) Geocodificar os municípios de interesse (PE) → lat/lon
# Usa Nominatim (OpenStreetMap): gratuito, mas exige User-Agent e máx. 1 req/s.
import time

municipios_full = [
    "Afogados da Ingazeira", "Afrânio", "Águas Belas", "Alagoinha", "Araripina",
    "Arcoverde", "Belém do São Francisco", "Belo Jardim", "Betânia", "Bodocó",
    "Bom Conselho", "Brejinho", "Brejo da Madre de Deus", "Buíque", "Cabrobó",
    "Caetés", "Calumbi", "Capoeiras", "Carnaíba", "Carnaubeira da Penha", "Cedro",
    "Custódia", "Dormentes", "Exu", "Flores", "Floresta", "Granito", "Iati",
    "Ibimirim", "Iguaracy", "Ingazeira", "Ipubi", "Itaíba", "Itapetim", "Jataúba",
    "Jatobá", "Lagoa Grande", "Manari", "Mirandiba", "Moreilândia", "Orocó",
    "Ouricuri", "Paranatama", "Parnamirim", "Pedra", "Pesqueira", "Petrolândia",
    "Petrolina", "Poção", "Quixaba", "Salgueiro", "Saloá", "Sanharó", "Santa Cruz",
    "Santa Cruz da Baixa Verde", "Santa Cruz do Capibaribe", "Santa Filomena",
    "Santa Maria da Boa Vista", "Santa Terezinha", "São Bento do Una",
    "São José do Belmonte", "Serra Talhada", "Serrita", "Sertânia", "Solidão",
    "Tabira", "Tacaratu", "Taquaritinga do Norte", "Terra Nova", "Trindade",
    "Triunfo", "Tupanatinga", "Tuparetama", "Venturosa", "Verdejante",
]

# --- AMOSTRA (10 municípios espalhados pelas regiões, pra iterar rápido) ---
# Troque `municipios = AMOSTRA` por `municipios = municipios_full` pra rodar as 75.
AMOSTRA = [
    "Petrolina", "Araripina", "Salgueiro", "Serra Talhada", "Arcoverde",
    "Ouricuri", "Custódia", "Ibimirim", "Santa Cruz do Capibaribe", "Buíque",
]
municipios = AMOSTRA

def geocode(nome, session):
    r = session.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": f"{nome}, Pernambuco, Brasil", "format": "json", "limit": 1},
        headers={"User-Agent": "quantum-ibf/1.0 (castrobeneyto@gmail.com)"},
        timeout=30,
    )
    r.raise_for_status()
    res = r.json()
    return (float(res[0]["lat"]), float(res[0]["lon"])) if res else None

session = requests.Session()
coords, falhas = {}, []
for i, nome in enumerate(municipios, 1):
    try:
        c = geocode(nome, session)
    except Exception as e:
        c = None
        print(f"  erro em {nome}: {e}")
    (coords.__setitem__(nome, c) if c else falhas.append(nome))
    print(f"[{i}/{len(municipios)}] {nome}: {c}")
    time.sleep(1.1)  # respeita o limite de 1 req/s do Nominatim

if falhas:
    print("\n⚠️ Não geocodificados (revise o nome):", falhas)

cidades = pd.DataFrame(
    [(n, lat, lon) for n, (lat, lon) in coords.items()],
    columns=["municipio", "lat", "lon"],
)
print(f"\n{len(cidades)} municípios geocodificados")
cidades.head()


In [ ]:
# 5) NASA POWER para cada município → um único DataFrame com a coluna 'municipio'
RENAME = {
    "PRECTOTCORR": "chuva_mm", "T2M": "temp_media", "T2M_MAX": "temp_max",
    "T2M_MIN": "temp_min", "RH2M": "umidade_rel",
}

def fetch_power(lat, lon, start="20100101", end="20231231"):
    resp = requests.get(
        "https://power.larc.nasa.gov/api/temporal/daily/point",
        params={
            "parameters": ",".join(RENAME), "community": "AG",
            "latitude": lat, "longitude": lon,
            "start": start, "end": end, "format": "JSON",
        },
        timeout=120,
    ).json()
    d = pd.DataFrame(resp["properties"]["parameter"])
    d.index = pd.to_datetime(d.index, format="%Y%m%d")
    d.index.name = "data"
    return d.rename(columns=RENAME).replace(-999, pd.NA).reset_index()

frames, erros = [], []
for i, row in cidades.iterrows():
    try:
        d = fetch_power(row.lat, row.lon)
        d.insert(1, "municipio", row.municipio)  # coluna da região que o ponto representa
        d["lat"], d["lon"] = row.lat, row.lon
        frames.append(d)
        print(f"[{i+1}/{len(cidades)}] {row.municipio}: {len(d)} dias")
    except Exception as e:
        erros.append((row.municipio, str(e)))
        print(f"[{i+1}/{len(cidades)}] {row.municipio}: ERRO {e}")
    time.sleep(0.3)

clima = pd.concat(frames, ignore_index=True)
if erros:
    print("\n⚠️ Municípios com erro:", erros)
print(f"\nDataFrame final: {clima.shape}")
clima.head()


In [ ]:
# 6) Salvar o dataset
from pathlib import Path

out = Path("dados")
out.mkdir(exist_ok=True)

clima.to_csv(out / "clima_pe_diario.csv", index=False)
try:
    clima.to_parquet(out / "clima_pe_diario.parquet", index=False)  # requer: uv add pyarrow
    print("Salvo CSV + Parquet em", out.resolve())
except Exception as e:
    print("CSV salvo em", out.resolve(), "| Parquet pulado (instale pyarrow):", e)


In [ ]:
# 7) Enriquecer 'cidades' com código IBGE + região (meso/microrregião)
# A API de Localidades do IBGE é estável. O código IBGE de 7 dígitos é a
# CHAVE pra juntar com qualquer base de vulnerabilidade/risco depois.
import unicodedata

def _norm(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.lower().strip()

munis_pe = get_json("https://servicodados.ibge.gov.br/api/v1/localidades/estados/26/municipios")
por_nome = {_norm(m["nome"]): m for m in munis_pe}

def info_ibge(nome):
    m = por_nome.get(_norm(nome))
    if not m:
        return {"cod_ibge": pd.NA, "microrregiao": pd.NA, "mesorregiao": pd.NA}
    return {
        "cod_ibge": str(m["id"]),
        "microrregiao": m["microrregiao"]["nome"],
        "mesorregiao": m["microrregiao"]["mesorregiao"]["nome"],
    }

extra = pd.DataFrame([info_ibge(n) for n in cidades["municipio"]], index=cidades.index)
cidades = pd.concat([cidades.drop(columns=extra.columns, errors="ignore"), extra], axis=1)

faltando = cidades.loc[cidades["cod_ibge"].isna(), "municipio"].tolist()
print("sem correspondência no IBGE (revisar nome):", faltando or "nenhuma")
print(cidades["mesorregiao"].value_counts())
cidades.head()


In [ ]:
# 7b) População (IBGE, estimativa 2024) -> exposição / porte do município
# Verificado via curl: agregado 6579 (estimativas de população), variável 9324.
# Uma única chamada em lote com todos os códigos IBGE.
codigos = cidades["cod_ibge"].dropna().astype(str).tolist()
loc = "N6[" + ",".join(codigos) + "]"
pop_json = get_json(
    f"https://servicodados.ibge.gov.br/api/v3/agregados/6579/"
    f"periodos/2024/variaveis/9324?localidades={loc}"
)
series = pop_json[0]["resultados"][0]["series"]
pop = {s["localidade"]["id"]: int(list(s["serie"].values())[0]) for s in series}
cidades["populacao_2024"] = cidades["cod_ibge"].map(pop)

print(f"população somada das {len(cidades)} cidades: {cidades['populacao_2024'].sum():,.0f} hab")
(cidades[["municipio", "cod_ibge", "mesorregiao", "populacao_2024"]]
 .sort_values("populacao_2024", ascending=False)
 .head(10))


In [ ]:
# 7c) PIB, área e densidade (IBGE) -> economia + exposição territorial
# Verificado via curl. PIB: agregado 5938 (2021, "Mil Reais");
# área/densidade: agregado 4714 (Censo 2022).
codigos = cidades["cod_ibge"].dropna().astype(str).tolist()
loc = "N6[" + ",".join(codigos) + "]"

def series_ibge(url):
    """{nome_variavel: {cod_ibge: valor_float}} para o período retornado."""
    out = {}
    for var in get_json(url):
        col = {}
        for s in var["resultados"][0]["series"]:
            v = list(s["serie"].values())[0]
            col[s["localidade"]["id"]] = pd.to_numeric(v, errors="coerce")
        out[var["variavel"]] = col
    return out

# PIB (mil reais, 2021)
pib = series_ibge(
    f"https://servicodados.ibge.gov.br/api/v3/agregados/5938/"
    f"periodos/2021/variaveis/37?localidades={loc}"
)
cidades["pib_milreais_2021"] = cidades["cod_ibge"].map(next(iter(pib.values())))

# Área (km²) + densidade (hab/km²) — Censo 2022
ad = series_ibge(
    f"https://servicodados.ibge.gov.br/api/v3/agregados/4714/"
    f"periodos/2022/variaveis/6318|614?localidades={loc}"
)
cidades["area_km2"] = cidades["cod_ibge"].map(ad["Área da unidade territorial"])
cidades["densidade_hab_km2"] = cidades["cod_ibge"].map(ad["Densidade demográfica"])

# PIB per capita (R$) = PIB(mil reais) * 1000 / população
cidades["pib_per_capita"] = (
    cidades["pib_milreais_2021"] * 1000 / cidades["populacao_2024"]
).round(0)

(cidades[["municipio", "populacao_2024", "area_km2", "densidade_hab_km2",
          "pib_milreais_2021", "pib_per_capita"]]
 .sort_values("pib_per_capita"))


In [ ]:
# 8) Agregar o clima diário -> MENSAL por município (features de seca)
# chuva = soma do mês (mm); temperaturas/umidade = média do mês.
# pd.Grouper (em vez de set_index+resample) p/ compatibilidade com pandas 3.0.
num_cols = ["chuva_mm", "temp_media", "temp_max", "temp_min", "umidade_rel"]
clima[num_cols] = clima[num_cols].apply(pd.to_numeric, errors="coerce")
clima["data"] = pd.to_datetime(clima["data"])

mensal = (
    clima.groupby(["municipio", pd.Grouper(key="data", freq="MS")])[num_cols]
    .agg({"chuva_mm": "sum", "temp_media": "mean", "temp_max": "mean",
          "temp_min": "mean", "umidade_rel": "mean"})
    .reset_index()
)

# cola código IBGE + região + população
mensal = mensal.merge(
    cidades[["municipio", "cod_ibge", "mesorregiao", "microrregiao", "populacao_2024", "lat", "lon"]],
    on="municipio", how="left",
)

mensal.to_csv(out / "clima_pe_mensal.csv", index=False)
print(f"{mensal.shape[0]} linhas (município × mês) — salvo em {(out / 'clima_pe_mensal.csv').resolve()}")
mensal.head()


In [ ]:
# 8b) Features de aridez / seca por município (derivadas só do clima)
# Índice de aridez de De Martonne = P_anual / (T_media_anual + 10).
#   < 10  árido | 10–20 semiárido | 20–30 subúmido | > 30 úmido
# "mês seco" = chuva mensal < 30 mm (limiar usual pra semiárido).
mensal["ano"] = mensal["data"].dt.year

anual = (
    mensal.groupby(["municipio", "ano"])
    .agg(chuva_anual=("chuva_mm", "sum"),
         temp_media_anual=("temp_media", "mean"),
         meses_secos=("chuva_mm", lambda s: int((s < 30).sum())),
         n_meses=("chuva_mm", "size"))
    .reset_index()
)
anual = anual[anual["n_meses"] == 12]  # só anos completos, pra não enviesar

resumo = (
    anual.groupby("municipio")
    .agg(chuva_anual_media=("chuva_anual", "mean"),
         chuva_anual_cv=("chuva_anual", lambda s: s.std() / s.mean()),  # variabilidade interanual
         temp_media_anual=("temp_media_anual", "mean"),
         meses_secos_ano=("meses_secos", "mean"))
    .reset_index()
)
resumo["indice_aridez"] = (resumo["chuva_anual_media"] / (resumo["temp_media_anual"] + 10)).round(1)

# Perfil final por município: socioeconômico (cidades) + aridez (resumo)
perfil = cidades.merge(resumo, on="municipio", how="left")
perfil.to_csv(out / "perfil_municipios_pe.csv", index=False)
print(f"Perfil salvo em {(out / 'perfil_municipios_pe.csv').resolve()}")

(perfil[["municipio", "mesorregiao", "populacao_2024", "pib_per_capita",
         "chuva_anual_media", "meses_secos_ano", "indice_aridez"]]
 .sort_values("indice_aridez"))


In [ ]:
# 9) Vulnerabilidade / risco por município (chave = cod_ibge)
#
# JÁ TEMOS (via API, dados reais no dataframe 'cidades'):
#   - populacao_2024 (IBGE)   -> exposição / porte
#   - mesorregiao/microrregiao (IBGE)
#
# FALTAM (baixar o arquivo e juntar por cod_ibge; sem API estável no momento):
#   IPEA IVS  -> Índice de Vulnerabilidade Social por município
#               http://ivs.ipea.gov.br  (base CSV/XLSX)
#   IDHM      -> http://atlasbrasil.org.br
#   CadÚnico  -> famílias em pobreza por município
#               https://aplicacoes.cidadania.gov.br/vis/data3
#   S2iD      -> decretos de emergência por estiagem (LABEL de impacto)
#
#   AdaptaBrasil (risco de seca decomposto em ameaça x exposição x vulnerabilidade):
#     TEM API REST (https://sistema.adaptabrasil.mcti.gov.br/api/...), mas em
#     2026-07 está retornando erro interno do servidor deles
#     ("SequenceGeneratorLimitExceeded"). Revisitar depois — é a MELHOR fonte
#     pro IBF quando voltar. Enquanto isso, dá pra baixar as camadas pelo portal.
#
# Padrão de join (quando tiver o CSV):
#   vuln = pd.read_csv("dados/ivs_municipios.csv", dtype={"cod_ibge": str})
#   base_ibf = mensal.merge(vuln, on="cod_ibge", how="left")  # município × mês + vulnerabilidade
#   base_ibf.to_csv(out / "base_ibf_pe.csv", index=False)

print("Base atual por município (dados reais já puxados):")
(cidades[["municipio", "cod_ibge", "mesorregiao", "populacao_2024", "lat", "lon"]]
 .sort_values("populacao_2024", ascending=False)
 .to_string(index=False))
